# Machine Unlearning – Gradient Ascent (Full-Window / Masked)

Finale 16-Window-Pipeline für den Vergleich von Full-Window- und token-selektivem Gradient Ascent. Neben Fakten- und Utility-Metriken werden tokenbasierte Forget-Set-Losses, Checkpoints und Effizienzmetriken in einem gemeinsamen Long/Tidy-Schema gespeichert.

Der Modus wird zentral über `UNLEARNING_CONFIG["loss_mode"]` gewählt: `"full_window"` für sequenzweiten und `"masked"` für token-selektiven Gradient Ascent.


In [ ]:
!pip -q install tiktoken

## 1. Google Drive einbinden

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Importe, Seed und Gerät

In [ ]:
import json
import math
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from IPython.display import display
from torch.utils.data import Dataset, DataLoader

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch-Version: {torch.__version__}")
print(f"Verwendetes Gerät: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 3. Projekt-, Modell- und Unlearning-Konfiguration

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/machine_unlearning_experiment")
BENCHMARK_DIR = PROJECT_DIR / "benchmark"
UNLEARNING_RUNS_DIR = PROJECT_DIR / "unlearning_runs"

FACTS_PATH = BENCHMARK_DIR / "facts_pakistan-islamabad_final.json"
EXPERIMENTS_PATH = BENCHMARK_DIR / "experiment_pakistan_capital_islamabad_final.json"
FORGET_SET_PATH = BENCHMARK_DIR / "forget_set_pakistan_islamabad_final.json"
MASK_ANNOTATIONS_PATH = BENCHMARK_DIR / "forget_mask_annotations_pakistan_islamabad_final.json"

BASE_MODEL_CHECKPOINT = Path(
    "/content/drive/MyDrive/model_checkpoints/"
    "clean_baseline_v1/model_final_step_0046460.pth"
)
TRAIN_TOKENS_PATH = Path("/content/drive/MyDrive/simplewiki_train_tokens_exact.pt")
VAL_TOKENS_PATH = Path("/content/drive/MyDrive/simplewiki_val_tokens_exact.pt")

SOURCE_MODEL_ID = "M0"
EXPERIMENT_ID = "pakistan_capital_islamabad_final_v1"

MODEL_CONFIG = {
    "vocab_size": 50_257,
    "context_length": 1_024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

UNLEARNING_CONFIG = {
    # Reproduzierbarer Umschalter: "full_window" oder "masked"
    "loss_mode": "masked",
    # Diese Lernraten wurden verwendet: 1e-06, 5e-06, 1e-05, 1.5e-05, 2e-05, 3e-05
    "learning_rate": 1e-5,
    "weight_decay": 0.0,
    "batch_size": 1,
    "gradient_clip_norm": 1.0,
    "max_steps": 16,
    "evaluation_steps": [0, 1, 2, 5, 10, 16],
    "checkpoint_steps": [1, 5, 10, 16],
}

LOSS_MODE = UNLEARNING_CONFIG["loss_mode"]
if LOSS_MODE not in {"full_window", "masked"}:
    raise ValueError(f"Unbekannter loss_mode: {LOSS_MODE!r}")
UNLEARNING_METHOD = f"gradient_ascent_{LOSS_MODE}"

VALIDATION_CONTEXT_LENGTH = 1024
VALIDATION_STRIDE = 1024
VALIDATION_BATCH_SIZE = 4
VALIDATION_EVAL_BATCHES = 50

# Optimizer-State ist bei AdamW sehr groß; standardmäßig nur Modell-Checkpoints.
SAVE_RESUME_CHECKPOINT = False

BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
UNLEARNING_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("M0:", BASE_MODEL_CHECKPOINT)
print("Forget-Set:", FORGET_SET_PATH)
print("Loss-Modus:", LOSS_MODE)
print("Masken-Datei:", MASK_ANNOTATIONS_PATH)


## 4. GPT-Modellarchitektur – unverändert aus der Trainingspipeline

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        if d_out % num_heads != 0:
            raise ValueError("d_out muss durch num_heads teilbar sein.")
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = queries @ keys.transpose(2, 3)
        causal_mask = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask, -torch.inf)
        attention_weights = torch.softmax(
            attention_scores / math.sqrt(self.head_dim), dim=-1
        )
        attention_weights = self.dropout(attention_weights)
        context = (attention_weights @ values).transpose(1, 2)
        context = context.reshape(batch_size, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1.0
            + torch.tanh(
                math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, sequence_length = in_idx.shape
        if sequence_length > self.pos_emb.num_embeddings:
            raise ValueError(
                f"Sequenzlänge {sequence_length} überschreitet "
                f"Context Length {self.pos_emb.num_embeddings}."
            )
        token_embeddings = self.tok_emb(in_idx)
        position_ids = torch.arange(sequence_length, device=in_idx.device)
        position_embeddings = self.pos_emb(position_ids)
        x = token_embeddings + position_embeddings
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

## 5. Dataset- und allgemeine Loss-Funktionen

In [ ]:
class CachedGPTDataset(Dataset):
    def __init__(self, token_tensor, max_length, stride):
        if token_tensor.ndim != 1:
            raise ValueError("token_tensor muss eindimensional sein.")
        if max_length <= 0:
            raise ValueError("max_length muss positiv sein.")
        if stride <= 0:
            raise ValueError("stride muss positiv sein.")
        if len(token_tensor) <= max_length:
            raise ValueError("Token-Tensor zu kurz für ein vollständiges Fenster.")
        self.token_tensor = token_tensor
        self.max_length = max_length
        self.stride = stride
        self.num_windows = 1 + (
            len(token_tensor) - max_length - 1
        ) // stride

    def __len__(self):
        return self.num_windows

    def __getitem__(self, index):
        if index < 0 or index >= self.num_windows:
            raise IndexError(index)
        start = index * self.stride
        input_ids = self.token_tensor[start : start + self.max_length]
        target_ids = self.token_tensor[start + 1 : start + self.max_length + 1]
        return input_ids, target_ids


class SelectedWindowDataset(Dataset):
    """Referenziert nur ausgewählte Fenster des originalen CachedGPTDataset."""
    def __init__(self, base_dataset, window_ids):
        self.base_dataset = base_dataset
        self.window_ids = [int(x) for x in window_ids]
        if not self.window_ids:
            raise ValueError("window_ids darf nicht leer sein.")
        if len(self.window_ids) != len(set(self.window_ids)):
            raise ValueError("window_ids enthält Duplikate.")
        for window_id in self.window_ids:
            if not 0 <= window_id < len(base_dataset):
                raise IndexError(f"Ungültige window_id: {window_id}")

    def __len__(self):
        return len(self.window_ids)

    def __getitem__(self, index):
        window_id = self.window_ids[index]
        input_ids, target_ids = self.base_dataset[window_id]
        return input_ids, target_ids, window_id


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


@torch.no_grad()
def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    batches_to_evaluate = (
        len(data_loader)
        if num_batches is None
        else min(num_batches, len(data_loader))
    )
    total_loss = 0.0
    for batch_index, (input_batch, target_batch) in enumerate(data_loader):
        if batch_index >= batches_to_evaluate:
            break
        total_loss += calc_loss_batch(
            input_batch, target_batch, model, device
        ).item()
    return total_loss / batches_to_evaluate


@torch.no_grad()
def evaluate_validation_loss(model, val_loader, device, eval_batches):
    was_training = model.training
    model.eval()
    try:
        return calc_loss_loader(
            val_loader, model, device, num_batches=eval_batches
        )
    finally:
        model.train(was_training)

## 6. Checkpoint-Hilfsfunktionen

In [ ]:
def load_checkpoint_file(checkpoint_path, map_location="cpu"):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint nicht gefunden: {checkpoint_path}")
    try:
        return torch.load(
            checkpoint_path, map_location=map_location, weights_only=False
        )
    except TypeError:
        return torch.load(checkpoint_path, map_location=map_location)


def looks_like_state_dict(candidate):
    return (
        isinstance(candidate, dict)
        and bool(candidate)
        and all(
            isinstance(k, str) and isinstance(v, torch.Tensor)
            for k, v in candidate.items()
        )
    )


def extract_model_state_dict(checkpoint):
    if looks_like_state_dict(checkpoint):
        return checkpoint
    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint ist weder State-Dict noch Dictionary.")
    for key in (
        "model_state_dict", "model_state", "state_dict", "model",
        "MODEL_STATE_DICT", "model_state_dict_cpu",
    ):
        candidate = checkpoint.get(key)
        if looks_like_state_dict(candidate):
            return candidate
    raise KeyError(f"Kein Modell-State-Dict. Schlüssel: {list(checkpoint.keys())}")


def strip_known_prefixes(state_dict):
    cleaned = dict(state_dict)
    for prefix in ("module.", "_orig_mod.", "model."):
        if cleaned and all(key.startswith(prefix) for key in cleaned):
            cleaned = {key[len(prefix):]: value for key, value in cleaned.items()}
    return cleaned


def load_model_from_checkpoint(checkpoint_path, model_config, device):
    checkpoint = load_checkpoint_file(checkpoint_path, map_location="cpu")
    state_dict = strip_known_prefixes(extract_model_state_dict(checkpoint))
    model = GPTModel(model_config)
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    del checkpoint, state_dict
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model


def save_unlearning_checkpoint(
    *, path, model, step, source_checkpoint, unlearning_config, forget_set_id
):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "unlearning_step": int(step),
            "source_checkpoint": str(source_checkpoint),
            "unlearning_method": UNLEARNING_METHOD,
            "unlearning_config": unlearning_config,
            "forget_set_id": forget_set_id,
            "seed": SEED,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        },
        path,
    )


def save_resume_checkpoint(
    *, path, model, optimizer, step, source_checkpoint,
    unlearning_config, forget_set_id
):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "unlearning_step": int(step),
        "source_checkpoint": str(source_checkpoint),
        "unlearning_method": UNLEARNING_METHOD,
        "unlearning_config": unlearning_config,
        "forget_set_id": forget_set_id,
        "seed": SEED,
        "torch_rng_state": torch.get_rng_state(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    if torch.cuda.is_available():
        payload["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    torch.save(payload, path)

## 7. Tokenizer und M0 laden

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
assert tokenizer.n_vocab == MODEL_CONFIG["vocab_size"]

MODEL = load_model_from_checkpoint(
    BASE_MODEL_CHECKPOINT, MODEL_CONFIG, DEVICE
)
parameter_count = sum(p.numel() for p in MODEL.parameters())

print(f"Tokenizer-Vokabular: {tokenizer.n_vocab:,}")
print(f"Parameter: {parameter_count:,}")
print("Model device:", next(MODEL.parameters()).device)

## 8. Benchmark laden und strukturell validieren

In [ ]:
def load_json(path):
    if not path.exists():
        raise FileNotFoundError(f"JSON-Datei nicht gefunden: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def validate_benchmark(facts_document, experiments_document):
    if "facts" not in facts_document:
        raise KeyError("In facts.json fehlt 'facts'.")
    if "experiments" not in experiments_document:
        raise KeyError("In experiments.json fehlt 'experiments'.")

    fact_ids = [f["fact_id"] for f in facts_document["facts"]]
    if len(fact_ids) != len(set(fact_ids)):
        raise ValueError("Doppelte fact_id-Werte.")
    fact_map = {f["fact_id"]: f for f in facts_document["facts"]}

    prompt_ids = []
    for fact in facts_document["facts"]:
        required = {"fact_id", "subject", "relation", "object", "prompts"}
        missing = required - fact.keys()
        if missing:
            raise KeyError(f"Fakt {fact.get('fact_id')}: fehlend {sorted(missing)}")
        for prompt in fact["prompts"]:
            required_prompt = {"prompt_id", "text", "target", "prompt_type"}
            missing_prompt = required_prompt - prompt.keys()
            if missing_prompt:
                raise KeyError(
                    f"Prompt {prompt.get('prompt_id')}: fehlend {sorted(missing_prompt)}"
                )
            prompt_ids.append(prompt["prompt_id"])

    if len(prompt_ids) != len(set(prompt_ids)):
        raise ValueError("Doppelte prompt_id-Werte.")

    for experiment in experiments_document["experiments"]:
        for group_name, ids in experiment["groups"].items():
            for fact_id in ids:
                if fact_id not in fact_map:
                    raise KeyError(
                        f"Unbekannte fact_id {fact_id!r} in {group_name!r}."
                    )

    print(
        f"Benchmark gültig: {len(fact_ids)} Fakten, "
        f"{len(prompt_ids)} Prompts."
    )


facts_document = load_json(FACTS_PATH)
experiments_document = load_json(EXPERIMENTS_PATH)
validate_benchmark(facts_document, experiments_document)

## 9. Token-Grenzen – inklusive Counterfactuals

In [ ]:
def inspect_token_boundaries(facts_document, tokenizer):
    rows = []
    for fact in facts_document["facts"]:
        for prompt in fact["prompts"]:
            targets = [("primary", prompt["target"])]
            targets.extend(
                ("alternative", x)
                for x in prompt.get("alternative_targets", [])
            )
            targets.extend(
                ("counterfactual", x)
                for x in prompt.get("counterfactual_targets", [])
            )
            for target_type, target in targets:
                p = tokenizer.encode(prompt["text"])
                t = tokenizer.encode(target)
                c = tokenizer.encode(prompt["text"] + target)
                rows.append({
                    "fact_id": fact["fact_id"],
                    "prompt_id": prompt["prompt_id"],
                    "target_type": target_type,
                    "target": target,
                    "stable_boundary": c == p + t,
                    "target_token_count": len(t),
                })
    return pd.DataFrame(rows)


boundary_report = inspect_token_boundaries(facts_document, tokenizer)
invalid_boundaries = boundary_report[~boundary_report["stable_boundary"]]
if invalid_boundaries.empty:
    print("Alle Prompt-Ziel-Paare besitzen stabile Token-Grenzen.")
else:
    display(invalid_boundaries)
    raise ValueError("Benchmark enthält instabile Token-Grenzen.")

## 10. Exakte Train- und Validation-Tokens laden

In [ ]:
train_data = torch.load(TRAIN_TOKENS_PATH, map_location="cpu")
val_data = torch.load(VAL_TOKENS_PATH, map_location="cpu")

TRAIN_TOKENS = train_data["train_tokens"]
VAL_TOKENS = val_data["val_tokens"]

assert train_data["context_length"] == MODEL_CONFIG["context_length"]
assert val_data["context_length"] == VALIDATION_CONTEXT_LENGTH
assert train_data["stride"] == 1024
assert val_data["stride"] == VALIDATION_STRIDE

print(f"Train-Tokens: {len(TRAIN_TOKENS):,}")
print(f"Validation-Tokens: {len(VAL_TOKENS):,}")

## 11. Forget-Set einmalig anlegen oder laden

Die 16 final manuell bestätigten `window_id`s bilden das Forget-Set. Existiert `FORGET_SET_PATH` bereits, bleibt `CREATE_FORGET_SET_FILE = False`.


In [ ]:
CREATE_FORGET_SET_FILE = False

FINAL_FORGET_WINDOW_IDS = [
    873, 3807, 6181, 6958, 9223, 9318, 9381, 9469,
    9485, 9937, 20056, 21737, 22589, 28362, 34359, 37114,
]


def write_forget_set_file(path, window_ids):
    if path.exists():
        raise FileExistsError(f"Forget-Set existiert bereits: {path}")

    window_ids = [int(x) for x in window_ids]
    if len(window_ids) != 16:
        raise ValueError(
            f"Exakt 16 Forget-Fenster erwartet, erhalten: {len(window_ids)}"
        )
    if len(window_ids) != len(set(window_ids)):
        raise ValueError("Forget-Liste enthält Duplikate.")

    document = {
        "forget_set_id": "pakistan_islamabad_explicit_current_windows_final",
        "fact_id": "pakistan_capital_islamabad",
        "selection_rule": (
            "Manually confirmed training windows that explicitly express "
            "the current relation capital(Pakistan, Islamabad)."
        ),
        "context_length": MODEL_CONFIG["context_length"],
        "stride": int(train_data["stride"]),
        "expected_window_count": 16,
        "windows": [
            {"window_id": x, "manual_label": "explicit_current"}
            for x in window_ids
        ],
    }

    with path.open("w", encoding="utf-8") as f:
        json.dump(document, f, ensure_ascii=False, indent=2)

    print("Forget-Set gespeichert:", path)


if CREATE_FORGET_SET_FILE:
    write_forget_set_file(FORGET_SET_PATH, FINAL_FORGET_WINDOW_IDS)
else:
    print("Finales Forget-Set wird nur geladen, nicht neu erzeugt.")


In [ ]:
def load_forget_set(path):
    document = load_json(path)

    if "windows" not in document:
        raise KeyError("Forget-Set enthält keinen Schlüssel 'windows'.")

    window_ids = [int(x["window_id"]) for x in document["windows"]]

    if document.get("forget_set_id") != (
        "pakistan_islamabad_explicit_current_windows_final"
    ):
        raise ValueError(
            "Unerwartete forget_set_id: "
            f"{document.get('forget_set_id')!r}"
        )

    if document.get("expected_window_count") != 16:
        raise ValueError(
            "Forget-Set deklariert nicht exakt 16 Fenster: "
            f"{document.get('expected_window_count')}"
        )

    if len(window_ids) != 16:
        raise ValueError(
            f"16 Forget-Fenster erwartet, erhalten: {len(window_ids)}"
        )

    if len(window_ids) != len(set(window_ids)):
        raise ValueError("Forget-Set enthält doppelte window_id-Werte.")

    if set(window_ids) != set(FINAL_FORGET_WINDOW_IDS):
        missing = sorted(set(FINAL_FORGET_WINDOW_IDS) - set(window_ids))
        extra = sorted(set(window_ids) - set(FINAL_FORGET_WINDOW_IDS))
        raise ValueError(
            "Forget-Set stimmt nicht mit FINAL_FORGET_WINDOW_IDS überein. "
            f"missing={missing}, extra={extra}"
        )

    if document.get("context_length") != MODEL_CONFIG["context_length"]:
        raise ValueError("Forget-Set context_length stimmt nicht mit Modell überein.")

    if document.get("stride") != int(train_data["stride"]):
        raise ValueError("Forget-Set stride stimmt nicht mit Training überein.")

    return document, window_ids


FORGET_DOCUMENT, FORGET_WINDOW_IDS = load_forget_set(FORGET_SET_PATH)
print("Forget-Set-ID:", FORGET_DOCUMENT["forget_set_id"])
print("Forget-Fenster:", len(FORGET_WINDOW_IDS))


## 11b. Mask-Annotationen für den token-selektiven Forget-Loss

Die 16 `window_id`s bleiben unverändert. Pro Evidenzfenster wird genau eine relationstragende Zielposition annotiert. Gespeichert werden `window_id`, `anchor_text` und `mask_text`; daraus wird die Position deterministisch im originalen 1024-Token-Fenster aufgelöst.


In [ ]:
def load_mask_annotations(path, expected_window_ids, expected_forget_set_id):
    document = load_json(path)
    annotations = document.get("annotations", [])

    if document.get("mask_set_id") != "pakistan_islamabad_masked_spans_final":
        raise ValueError(
            "Unerwartete mask_set_id: "
            f"{document.get('mask_set_id')!r}"
        )

    if document.get("forget_set_id") != expected_forget_set_id:
        raise ValueError(
            "forget_set_id von Masken- und Forget-Set-Datei stimmen nicht überein. "
            f"Maske={document.get('forget_set_id')!r}, "
            f"Forget-Set={expected_forget_set_id!r}"
        )

    if document.get("expected_window_count") != 16:
        raise ValueError(
            "Masken-Datei deklariert nicht exakt 16 Fenster: "
            f"{document.get('expected_window_count')}"
        )

    if len(annotations) != 16:
        raise ValueError(
            f"Exakt 16 Masken-Annotationen erwartet, erhalten: {len(annotations)}"
        )

    annotation_window_ids = [int(x["window_id"]) for x in annotations]
    if len(annotation_window_ids) != len(set(annotation_window_ids)):
        raise ValueError("Masken-Datei enthält doppelte window_id-Werte.")

    expected = set(int(x) for x in expected_window_ids)
    actual = set(annotation_window_ids)
    if expected != actual:
        missing = sorted(expected - actual)
        extra = sorted(actual - expected)
        raise ValueError(
            "Masken/Forget-Set stimmen nicht überein. "
            f"missing={missing}, extra={extra}"
        )

    by_window = {
        int(annotation["window_id"]): annotation
        for annotation in annotations
    }

    return document, by_window


MASK_DOCUMENT, MASK_ANNOTATIONS_BY_WINDOW = load_mask_annotations(
    MASK_ANNOTATIONS_PATH,
    FORGET_WINDOW_IDS,
    FORGET_DOCUMENT["forget_set_id"],
)

print("Mask-Set-ID:", MASK_DOCUMENT["mask_set_id"])
print("Annotierte Fenster:", len(MASK_ANNOTATIONS_BY_WINDOW))
print("✓ Finale 16er-Maskendatei erfolgreich validiert.")


## 12. Forget- und Validation-Loader erzeugen

In [ ]:
TRAIN_WINDOW_DATASET = CachedGPTDataset(
    TRAIN_TOKENS,
    max_length=MODEL_CONFIG["context_length"],
    stride=int(train_data["stride"]),
)

FORGET_DATASET = SelectedWindowDataset(
    TRAIN_WINDOW_DATASET,
    FORGET_WINDOW_IDS,
)

forget_generator = torch.Generator()
forget_generator.manual_seed(SEED)

FORGET_LOADER = DataLoader(
    FORGET_DATASET,
    batch_size=UNLEARNING_CONFIG["batch_size"],
    shuffle=True,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
    generator=forget_generator,
)

VAL_DATASET = CachedGPTDataset(
    VAL_TOKENS,
    max_length=VALIDATION_CONTEXT_LENGTH,
    stride=VALIDATION_STRIDE,
)

VAL_LOADER = DataLoader(
    VAL_DATASET,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

print(f"Trainingsfenster gesamt: {len(TRAIN_WINDOW_DATASET):,}")
print(f"Forget-Fenster: {len(FORGET_DATASET)}")
print(f"Validation-Fenster: {len(VAL_DATASET):,}")

## 12b. Masken gegen die originalen Target-Tokens auflösen

Das Modell bekommt weiterhin das vollständige Window als Input. Die Bool-Maske hat dieselbe Länge wie `target_ids` und markiert nur die Positionen, deren Cross-Entropy zum Forget-Loss beitragen soll.


### Robuste Maskenauflösung auf dem originalen Tokenstream

Die Textannotation wird **nicht erneut mit GPT-2-BPE tokenisiert**. Stattdessen wird der originale `target_ids`-Stream bytegenau rekonstruiert, der annotierte Textspan darin gesucht und anschließend auf die bereits vorhandenen Tokenpositionen zurückgeführt. Dadurch bleiben Whitespace- und BPE-Kontextgrenzen reproduzierbar.


In [ ]:
import re


def whitespace_flexible_pattern(text: str):
    """
    Erzeugt ein Regex, das den Text exakt bis auf Whitespace-Mengen matcht.

    Ein Leerzeichen in der Annotation matcht dadurch auch zwei Leerzeichen,
    Tabs, Zeilenumbrüche oder NBSP im ursprünglichen Trainingsstream.
    """
    parts = re.split(r"\s+", text.strip())
    return re.compile(r"\s+".join(re.escape(part) for part in parts))


def token_byte_boundaries(token_ids, tokenizer):
    """
    Rekonstruiert den ORIGINALEN GPT-2-Bytestream samt Token-Grenzen.
    """
    boundaries = [0]
    token_bytes = []
    total = 0

    for token_id in token_ids.tolist():
        piece = tokenizer.decode_single_token_bytes(int(token_id))
        token_bytes.append(piece)
        total += len(piece)
        boundaries.append(total)

    return b"".join(token_bytes), boundaries


def byte_span_to_token_positions(
    span_start,
    span_end,
    boundaries,
    window_id,
):
    """
    Liefert alle ORIGINAL-Target-Tokens, die den Textspan überdecken.

    GPT-2-Tokens enthalten häufig führenden Whitespace, z.B. " Pakistan".
    Deshalb verlangen wir nicht, dass der Textspan exakt an der
    Token-Grenze beginnt.
    """
    positions = []

    for i in range(len(boundaries) - 1):
        token_start = boundaries[i]
        token_end = boundaries[i + 1]

        if token_end > span_start and token_start < span_end:
            positions.append(i)

    if not positions:
        raise ValueError(
            f"Window {window_id}: Textspan [{span_start}, {span_end}) "
            "überlappt kein Target-Token."
        )

    return positions


def char_span_to_byte_span(
    text: str,
    char_start: int,
    char_end: int,
):
    """
    Mappt Python-Zeichenpositionen auf UTF-8-Bytepositionen.
    """
    byte_start = len(text[:char_start].encode("utf-8"))
    byte_end = len(text[:char_end].encode("utf-8"))

    return byte_start, byte_end


def resolve_window_loss_mask(
    target_ids,
    annotation,
    tokenizer,
):
    """
    Löst die Textannotation auf die ORIGINALEN Target-Tokenpositionen auf.

    Der Anchor wird whitespace-tolerant gesucht.
    Die finale Loss-Maske bleibt aber exakt auf den ursprünglichen
    GPT-2-Tokenpositionen des Trainingsstreams.
    """
    window_id = int(annotation["window_id"])

    target_bytes, boundaries = token_byte_boundaries(
        target_ids,
        tokenizer,
    )

    decoded_window = target_bytes.decode(
        "utf-8",
        errors="strict",
    )

    # ---------------------------------------------------------
    # 1. Anchor im Originaltext suchen.
    #    Whitespace darf variieren.
    # ---------------------------------------------------------
    anchor_pattern = whitespace_flexible_pattern(
        annotation["anchor_text"]
    )

    anchor_matches = list(
        anchor_pattern.finditer(decoded_window)
    )

    if len(anchor_matches) != 1:
        key = annotation["mask_text"].strip()
        nearby_index = (
            decoded_window.find(key)
            if key
            else -1
        )

        if nearby_index >= 0:
            left = max(0, nearby_index - 140)
            right = min(
                len(decoded_window),
                nearby_index + len(key) + 140,
            )

            nearby = decoded_window[left:right].replace(
                "\n",
                " ",
            )
        else:
            nearby = (
                "<mask_text selbst nicht im "
                "dekodierten Window gefunden>"
            )

        raise ValueError(
            f"Window {window_id}: whitespace-toleranter "
            f"anchor_text muss exakt einmal vorkommen, "
            f"gefunden={len(anchor_matches)}. "
            f"Anchor={annotation['anchor_text']!r}\n"
            f"Diagnose-Kontext: {nearby!r}"
        )

    anchor_match = anchor_matches[0]

    anchor_text_original = decoded_window[
        anchor_match.start():anchor_match.end()
    ]

    # ---------------------------------------------------------
    # 2. Maskentext innerhalb des gefundenen Anchors suchen.
    # ---------------------------------------------------------
    mask_pattern = whitespace_flexible_pattern(
        annotation["mask_text"]
    )

    mask_matches = list(
        mask_pattern.finditer(anchor_text_original)
    )

    if len(mask_matches) != 1:
        raise ValueError(
            f"Window {window_id}: mask_text muss innerhalb "
            f"des aufgelösten Anchors exakt einmal vorkommen, "
            f"gefunden={len(mask_matches)}. "
            f"Mask={annotation['mask_text']!r}, "
            f"resolved_anchor={anchor_text_original!r}"
        )

    mask_match = mask_matches[0]

    mask_char_start = (
        anchor_match.start()
        + mask_match.start()
    )

    mask_char_end = (
        anchor_match.start()
        + mask_match.end()
    )

    # ---------------------------------------------------------
    # 3. Zeichenposition -> Byteposition
    # ---------------------------------------------------------
    mask_byte_start, mask_byte_end = (
        char_span_to_byte_span(
            decoded_window,
            mask_char_start,
            mask_char_end,
        )
    )

    # ---------------------------------------------------------
    # 4. Byteposition -> originale Target-Tokenpositionen
    # ---------------------------------------------------------
    positions = byte_span_to_token_positions(
        span_start=mask_byte_start,
        span_end=mask_byte_end,
        boundaries=boundaries,
        window_id=window_id,
    )

    loss_mask = torch.zeros_like(
        target_ids,
        dtype=torch.bool,
    )

    loss_mask[positions] = True

    # ---------------------------------------------------------
    # 5. Sanity Check
    # ---------------------------------------------------------
    decoded_mask = tokenizer.decode(
        [
            int(target_ids[pos].item())
            for pos in positions
        ]
    )

    expected_content = annotation[
        "mask_text"
    ].strip()

    if decoded_mask.strip() != expected_content:
        raise ValueError(
            f"Window {window_id}: Die den Textspan "
            f"überdeckenden GPT-2-Tokens dekodieren zu "
            f"{decoded_mask!r}, erwartet war inhaltlich "
            f"{expected_content!r}. "
            "Bitte Annotation prüfen."
        )

    return (
        loss_mask,
        positions,
        anchor_match.start(),
        anchor_text_original,
    )


# =============================================================
# Masken für alle 16 Forget-Windows auflösen
# =============================================================
# WICHTIG:
# Die Masken werden in BEIDEN Modi aufgelöst. Im Full-Window-Modus
# beeinflussen sie den Trainingsloss NICHT; sie werden dort nur für die
# spätere tokenbasierte Selektivitäts-Evaluation benötigt.

WINDOW_LOSS_MASKS = {}
RESOLVED_MASK_ANNOTATIONS = []

for window_id in FORGET_WINDOW_IDS:
    _, target_ids = TRAIN_WINDOW_DATASET[window_id]

    annotation = MASK_ANNOTATIONS_BY_WINDOW[int(window_id)]

    (
        loss_mask,
        positions,
        anchor_char_start,
        resolved_anchor,
    ) = resolve_window_loss_mask(
        target_ids=target_ids,
        annotation=annotation,
        tokenizer=tokenizer,
    )

    WINDOW_LOSS_MASKS[int(window_id)] = loss_mask

    context_left = max(0, positions[0] - 12)
    context_right = min(len(target_ids), positions[-1] + 13)

    context_text = tokenizer.decode(
        target_ids[context_left:context_right].tolist()
    )

    RESOLVED_MASK_ANNOTATIONS.append({
        **annotation,
        "target_positions": [int(x) for x in positions],
        "target_token_ids": [
            int(target_ids[pos].item()) for pos in positions
        ],
        "target_tokens": [
            tokenizer.decode([int(target_ids[pos].item())])
            for pos in positions
        ],
        "resolved_context": context_text,
        "resolved_anchor": resolved_anchor,
        "anchor_char_start": int(anchor_char_start),
    })

assert len(WINDOW_LOSS_MASKS) == 16
assert set(WINDOW_LOSS_MASKS) == set(FORGET_WINDOW_IDS)

print(
    f"Masken erfolgreich aufgelöst: "
    f"{len(WINDOW_LOSS_MASKS)} Fenster"
)

for item in RESOLVED_MASK_ANNOTATIONS:
    print(
        f"Window {item['window_id']:>5} | "
        f"{item['manual_label']:<16} | "
        f"quality={item['annotation_quality']:<6} | "
        f"positions={item['target_positions']} | "
        f"token_ids={item['target_token_ids']} | "
        f"mask={item['mask_text']!r}"
    )


## 13. Forget-Set-Sanity-Check – vor dem ersten Update prüfen

In [ ]:
for index in range(min(3, len(FORGET_DATASET))):
    input_ids, _, window_id = FORGET_DATASET[index]
    print("\n" + "=" * 80)
    print(f"Forget Window {window_id}")
    print("=" * 80)
    print(tokenizer.decode(input_ids.tolist())[:2000])

## 14. Fakten-Evaluation – train/eval-Modus wird korrekt wiederhergestellt

In [ ]:
@torch.no_grad()
def score_target_sequence(model, tokenizer, prompt, target):
    was_training = model.training
    model.eval()
    try:
        device = next(model.parameters()).device
        prompt_ids = tokenizer.encode(prompt)
        target_ids = tokenizer.encode(target)
        combined_ids = tokenizer.encode(prompt + target)

        if not prompt_ids or not target_ids:
            raise ValueError("Prompt und Target dürfen nicht leer sein.")
        if combined_ids != prompt_ids + target_ids:
            raise ValueError(
                f"Instabile Token-Grenze. Prompt={prompt!r}, Target={target!r}"
            )

        input_ids = torch.tensor(
            combined_ids[:-1], dtype=torch.long, device=device
        ).unsqueeze(0)
        logits = model(input_ids)
        log_probs = F.log_softmax(logits, dim=-1)

        start_position = len(prompt_ids) - 1
        target_length = len(target_ids)
        relevant = log_probs[
            0, start_position : start_position + target_length, :
        ]
        target_tensor = torch.tensor(target_ids, dtype=torch.long, device=device)
        token_log_probs = relevant.gather(
            1, target_tensor.unsqueeze(1)
        ).squeeze(1)
        token_probabilities = token_log_probs.exp()

        sequence_nll = -token_log_probs.mean().item()
        total_sequence_nll = -token_log_probs.sum().item()
        first_token_logits = logits[0, start_position]
        first_id = target_ids[0]
        first_logit = first_token_logits[first_id]
        first_rank = int((first_token_logits > first_logit).sum().item()) + 1
        first_probability = torch.softmax(
            first_token_logits, dim=-1
        )[first_id].item()

        return {
            "sequence_nll": sequence_nll,
            "total_sequence_nll": total_sequence_nll,
            "geo_mean_probability": math.exp(-sequence_nll),
            "first_token_probability": first_probability,
            "first_token_rank": first_rank,
            "target_token_count": target_length,
            "target_token_ids": target_ids,
            "target_tokens": [tokenizer.decode([x]) for x in target_ids],
            "token_log_probs": token_log_probs.cpu().tolist(),
            "token_probabilities": token_probabilities.cpu().tolist(),
        }
    finally:
        model.train(was_training)


def get_experiment(experiments_document, experiment_id):
    for experiment in experiments_document["experiments"]:
        if experiment["experiment_id"] == experiment_id:
            return experiment
    raise KeyError(f"Experiment {experiment_id!r} nicht gefunden.")


def run_fact_evaluation(
    model, tokenizer, facts_document, experiments_document,
    experiment_id, model_id, checkpoint_label
):
    fact_map = {f["fact_id"]: f for f in facts_document["facts"]}
    experiment = get_experiment(experiments_document, experiment_id)
    rows = []

    for group_name, fact_ids in experiment["groups"].items():
        for fact_id in fact_ids:
            fact = fact_map[fact_id]
            for prompt in fact["prompts"]:
                targets = [("primary", prompt["target"])]
                targets.extend(
                    ("alternative", x)
                    for x in prompt.get("alternative_targets", [])
                )
                targets.extend(
                    ("counterfactual", x)
                    for x in prompt.get("counterfactual_targets", [])
                )
                for target_type, target in targets:
                    scores = score_target_sequence(
                        model, tokenizer, prompt["text"], target
                    )
                    rows.append({
                        "model_id": model_id,
                        "checkpoint": checkpoint_label,
                        "experiment_id": experiment_id,
                        "group": group_name,
                        "fact_id": fact_id,
                        "subject": fact["subject"],
                        "relation": fact["relation"],
                        "object": fact["object"],
                        "prompt_id": prompt["prompt_id"],
                        "prompt_type": prompt["prompt_type"],
                        "orientation": prompt.get("orientation"),
                        "prompt": prompt["text"],
                        "target": target,
                        "target_type": target_type,
                        **scores,
                    })
    return pd.DataFrame(rows)

## 15. Aggregation und M0-vs.-GA-Vergleich

In [ ]:
def derive_analysis_group(row):
    group = row["group"]
    if group not in {"target", "same_relation"}:
        return group

    orientation = row.get("orientation")
    if orientation == "country_to_capital":
        return f"{group}_direct"
    if orientation == "capital_to_country":
        return f"{group}_inverse"

    prompt_type = str(row.get("prompt_type", "")).lower()
    if "inverse" in prompt_type:
        return f"{group}_inverse"
    if "direct" in prompt_type or "possessive" in prompt_type:
        return f"{group}_direct"

    raise ValueError("Prompt-Richtung konnte nicht bestimmt werden.")


def aggregate_fact_results(results):
    primary = results[results["target_type"] == "primary"].copy()

    if "analysis_group" not in primary.columns:
        primary["analysis_group"] = primary.apply(
            derive_analysis_group,
            axis=1,
        )

    fact_summary = (
        primary.groupby(
            [
                "group", "analysis_group", "fact_id",
                "subject", "relation", "object",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            mean_sequence_nll=("sequence_nll", "mean"),
            std_sequence_nll=("sequence_nll", "std"),
            mean_geo_probability=("geo_mean_probability", "mean"),
            mean_first_token_probability=("first_token_probability", "mean"),
            median_first_token_rank=("first_token_rank", "median"),
            max_first_token_rank=("first_token_rank", "max"),
            prompt_count=("prompt_id", "nunique"),
            target_token_count=("target_token_count", "max"),
        )
    )
    fact_summary["std_sequence_nll"] = (
        fact_summary["std_sequence_nll"].fillna(0.0)
    )

    group_summary = (
        fact_summary.groupby("analysis_group", as_index=False)
        .agg(
            fact_count=("fact_id", "nunique"),
            prompt_count=("prompt_count", "sum"),
            mean_fact_nll=("mean_sequence_nll", "mean"),
            std_fact_nll=("mean_sequence_nll", "std"),
            median_fact_rank=("median_first_token_rank", "median"),
            mean_fact_geo_probability=("mean_geo_probability", "mean"),
            mean_fact_first_token_probability=(
                "mean_first_token_probability",
                "mean",
            ),
        )
    )
    group_summary["std_fact_nll"] = (
        group_summary["std_fact_nll"].fillna(0.0)
    )

    return primary, fact_summary, group_summary


def compare_evaluation_runs(baseline_results, candidate_results):
    """
    Prompt-genauer Vergleich in einem stabilen Long/Tidy-Schema.

    Anders als zuvor entstehen KEINE step-spezifischen Spalten wie
    sequence_nll_GA1, sequence_nll_GA2 usw. Jeder Kandidaten-Step benutzt
    dieselben Spaltennamen und kann anschließend gefahrlos per pd.concat
    zusammengeführt werden.
    """
    join_columns = [
        "experiment_id",
        "group",
        "fact_id",
        "prompt_id",
        "target_type",
        "target",
    ]

    metric_columns = [
        "sequence_nll",
        "total_sequence_nll",
        "geo_mean_probability",
        "first_token_rank",
        "first_token_probability",
    ]

    baseline = baseline_results[join_columns + metric_columns].rename(
        columns={
            metric: f"baseline_{metric}"
            for metric in metric_columns
        }
    )

    candidate = candidate_results[join_columns + metric_columns].rename(
        columns={
            metric: f"candidate_{metric}"
            for metric in metric_columns
        }
    )

    comparison = baseline.merge(
        candidate,
        on=join_columns,
        how="inner",
        validate="one_to_one",
    )

    if len(comparison) != len(baseline_results):
        raise ValueError(
            "Baseline wurde nicht vollständig gematcht: "
            f"{len(comparison)} von {len(baseline_results)} Zeilen."
        )

    if len(comparison) != len(candidate_results):
        raise ValueError(
            "Candidate wurde nicht vollständig gematcht: "
            f"{len(comparison)} von {len(candidate_results)} Zeilen."
        )

    comparison["delta_sequence_nll"] = (
        comparison["candidate_sequence_nll"]
        - comparison["baseline_sequence_nll"]
    )
    comparison["delta_total_sequence_nll"] = (
        comparison["candidate_total_sequence_nll"]
        - comparison["baseline_total_sequence_nll"]
    )
    comparison["delta_first_token_rank"] = (
        comparison["candidate_first_token_rank"]
        - comparison["baseline_first_token_rank"]
    )
    comparison["delta_geo_mean_probability"] = (
        comparison["candidate_geo_mean_probability"]
        - comparison["baseline_geo_mean_probability"]
    )
    comparison["delta_first_token_probability"] = (
        comparison["candidate_first_token_probability"]
        - comparison["baseline_first_token_probability"]
    )
    comparison["geo_probability_ratio"] = (
        comparison["candidate_geo_mean_probability"]
        / comparison["baseline_geo_mean_probability"].clip(lower=1e-30)
    )
    comparison["first_token_probability_ratio"] = (
        comparison["candidate_first_token_probability"]
        / comparison["baseline_first_token_probability"].clip(lower=1e-30)
    )

    return comparison


## 16. Unlearning-Run-Verzeichnis und Snapshots anlegen

In [ ]:
RUN_WALL_START_TIME = time.perf_counter()

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
learning_rate_tag = format(float(UNLEARNING_CONFIG["learning_rate"]), ".6g")

UNLEARNING_RUN_NAME = (
    f"ga_{LOSS_MODE}_lr_{learning_rate_tag}_"
    f"{EXPERIMENT_ID}_{run_timestamp}"
)
UNLEARNING_RUN_DIR = UNLEARNING_RUNS_DIR / UNLEARNING_RUN_NAME
CHECKPOINT_DIR = UNLEARNING_RUN_DIR / "checkpoints"
EVALUATION_DIR = UNLEARNING_RUN_DIR / "evaluations"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=False)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)


def write_json(path, document):
    with path.open("w", encoding="utf-8") as f:
        json.dump(document, f, ensure_ascii=False, indent=2)


COMMON_RUN_METADATA = {
    "run_id": UNLEARNING_RUN_NAME,
    "source_model_id": SOURCE_MODEL_ID,
    "experiment_id": EXPERIMENT_ID,
    "benchmark_version": facts_document.get("benchmark_version", "unknown"),
    "forget_set_id": FORGET_DOCUMENT["forget_set_id"],
    "mask_set_id": MASK_DOCUMENT["mask_set_id"],
    "unlearning_method": UNLEARNING_METHOD,
    "loss_mode": LOSS_MODE,
    "learning_rate": float(UNLEARNING_CONFIG["learning_rate"]),
    "weight_decay": float(UNLEARNING_CONFIG["weight_decay"]),
    "batch_size": int(UNLEARNING_CONFIG["batch_size"]),
    "gradient_clip_norm": UNLEARNING_CONFIG["gradient_clip_norm"],
    "max_steps": int(UNLEARNING_CONFIG["max_steps"]),
    "seed": int(SEED),
}


def add_run_metadata(dataframe, *, step=None):
    """Fügt Run-Metadaten als echte Tabellenspalten hinzu."""
    result = dataframe.copy()

    metadata = dict(COMMON_RUN_METADATA)
    if step is not None:
        metadata["unlearning_step"] = int(step)

    for key, value in metadata.items():
        result[key] = value

    metadata_columns = list(metadata)
    other_columns = [
        column for column in result.columns
        if column not in metadata_columns
    ]

    return result[metadata_columns + other_columns]


write_json(UNLEARNING_RUN_DIR / "facts_snapshot.json", facts_document)
write_json(UNLEARNING_RUN_DIR / "experiments_snapshot.json", experiments_document)
write_json(UNLEARNING_RUN_DIR / "forget_set_snapshot.json", FORGET_DOCUMENT)
write_json(UNLEARNING_RUN_DIR / "mask_annotations_snapshot.json", MASK_DOCUMENT)
write_json(
    UNLEARNING_RUN_DIR / "resolved_mask_annotations.json",
    {
        "mask_set_id": MASK_DOCUMENT["mask_set_id"],
        "annotations": RESOLVED_MASK_ANNOTATIONS,
    },
)
write_json(
    UNLEARNING_RUN_DIR / "unlearning_config.json",
    {
        **COMMON_RUN_METADATA,
        "source_checkpoint": str(BASE_MODEL_CHECKPOINT),
        "unlearning_config": UNLEARNING_CONFIG,
        "validation": {
            "eval_batches": VALIDATION_EVAL_BATCHES,
            "context_length": VALIDATION_CONTEXT_LENGTH,
            "stride": VALIDATION_STRIDE,
            "batch_size": VALIDATION_BATCH_SIZE,
        },
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    },
)

print("Run:", UNLEARNING_RUN_DIR)
print("Learning Rate:", UNLEARNING_CONFIG["learning_rate"])


## 17. Einheitliche Fakten- und Utility-Evaluation pro Modellzustand

In [ ]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


@torch.no_grad()
def evaluate_forget_set_losses(
    *,
    model,
    forget_dataset,
    window_loss_masks,
    device,
):
    """
    Wertet IMMER alle 16 Forget-Windows aus und trennt drei Bereiche:

    1. full_window_nll:
       mittlere NLL über sämtliche 1024 Targetpositionen je Window.
    2. relation_target_nll:
       mittlere NLL ausschließlich über die annotierten relationstragenden
       Targetpositionen.
    3. non_target_context_nll:
       mittlere NLL über alle übrigen Positionen derselben Forget-Windows.

    Die Aggregation ist tokengewichtet; zusätzlich werden per-Window-Werte
    gespeichert, damit spätere Analysen neu aggregiert werden können.
    """
    was_training = model.training
    model.eval()

    total_full_loss = 0.0
    total_relation_loss = 0.0
    total_context_loss = 0.0

    total_full_tokens = 0
    total_relation_tokens = 0
    total_context_tokens = 0

    window_rows = []

    try:
        for dataset_index in range(len(forget_dataset)):
            input_ids, target_ids, window_id = forget_dataset[dataset_index]
            window_id = int(window_id)

            input_batch = input_ids.unsqueeze(0).to(
                device,
                non_blocking=True,
            )
            target_batch = target_ids.unsqueeze(0).to(
                device,
                non_blocking=True,
            )

            logits = model(input_batch)
            token_losses = F.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten(),
                reduction="none",
            ).view(-1)

            relation_mask = window_loss_masks[window_id].to(
                device=device,
                dtype=torch.bool,
            )

            if relation_mask.shape != token_losses.shape:
                raise ValueError(
                    f"Window {window_id}: Maskenform "
                    f"{tuple(relation_mask.shape)} != "
                    f"Lossform {tuple(token_losses.shape)}"
                )

            context_mask = ~relation_mask

            relation_tokens = int(relation_mask.sum().item())
            context_tokens = int(context_mask.sum().item())
            full_tokens = int(token_losses.numel())

            if relation_tokens == 0:
                raise ValueError(
                    f"Window {window_id}: keine relationstragenden Tokens."
                )
            if context_tokens == 0:
                raise ValueError(
                    f"Window {window_id}: keine Context-Tokens."
                )

            full_sum = float(token_losses.sum().item())
            relation_sum = float(token_losses[relation_mask].sum().item())
            context_sum = float(token_losses[context_mask].sum().item())

            total_full_loss += full_sum
            total_relation_loss += relation_sum
            total_context_loss += context_sum

            total_full_tokens += full_tokens
            total_relation_tokens += relation_tokens
            total_context_tokens += context_tokens

            window_rows.append({
                "window_id": window_id,
                "full_window_nll": full_sum / full_tokens,
                "relation_target_nll": relation_sum / relation_tokens,
                "non_target_context_nll": context_sum / context_tokens,
                "full_window_token_count": full_tokens,
                "relation_target_token_count": relation_tokens,
                "non_target_context_token_count": context_tokens,
            })

        summary = pd.DataFrame([{
            "forget_window_count": len(forget_dataset),
            "full_window_nll": total_full_loss / total_full_tokens,
            "relation_target_nll": (
                total_relation_loss / total_relation_tokens
            ),
            "non_target_context_nll": (
                total_context_loss / total_context_tokens
            ),
            "full_window_token_count": total_full_tokens,
            "relation_target_token_count": total_relation_tokens,
            "non_target_context_token_count": total_context_tokens,
        }])

        return summary, pd.DataFrame(window_rows)

    finally:
        model.train(was_training)


def evaluate_and_save_model_state(*, model, step, checkpoint_label):
    model_id = SOURCE_MODEL_ID if step == 0 else f"GA_step_{step:04d}"
    state_dir = EVALUATION_DIR / f"step_{step:04d}"
    state_dir.mkdir(parents=True, exist_ok=False)

    # ---------------------------------------------------------
    # 1. Fakten-/Prompt-Evaluation
    # ---------------------------------------------------------
    synchronize_device()
    fact_eval_start = time.perf_counter()

    results = run_fact_evaluation(
        model=model,
        tokenizer=tokenizer,
        facts_document=facts_document,
        experiments_document=experiments_document,
        experiment_id=EXPERIMENT_ID,
        model_id=model_id,
        checkpoint_label=checkpoint_label,
    )

    synchronize_device()
    fact_evaluation_seconds = time.perf_counter() - fact_eval_start

    results["analysis_group"] = results.apply(
        derive_analysis_group,
        axis=1,
    )
    results = add_run_metadata(results, step=step)

    primary, fact_summary, group_summary = aggregate_fact_results(results)
    fact_summary = add_run_metadata(fact_summary, step=step)
    group_summary = add_run_metadata(group_summary, step=step)

    alternative = results[results["target_type"] == "alternative"].copy()
    counterfactual = results[results["target_type"] == "counterfactual"].copy()

    # ---------------------------------------------------------
    # 2. Tokenbasierte Forget-Set-Evaluation
    # ---------------------------------------------------------
    synchronize_device()
    forget_eval_start = time.perf_counter()

    forget_set_summary, forget_window_losses = evaluate_forget_set_losses(
        model=model,
        forget_dataset=FORGET_DATASET,
        window_loss_masks=WINDOW_LOSS_MASKS,
        device=DEVICE,
    )

    synchronize_device()
    forget_set_evaluation_seconds = time.perf_counter() - forget_eval_start

    forget_set_summary = add_run_metadata(
        forget_set_summary,
        step=step,
    )
    forget_window_losses = add_run_metadata(
        forget_window_losses,
        step=step,
    )

    # ---------------------------------------------------------
    # 3. Globale Utility auf unveränderter Validation
    # ---------------------------------------------------------
    synchronize_device()
    validation_start = time.perf_counter()

    validation_loss = evaluate_validation_loss(
        model,
        VAL_LOADER,
        DEVICE,
        VALIDATION_EVAL_BATCHES,
    )

    synchronize_device()
    validation_evaluation_seconds = time.perf_counter() - validation_start

    utility_summary = add_run_metadata(
        pd.DataFrame([{
            "model_id": model_id,
            "checkpoint": checkpoint_label,
            "validation_loss": float(validation_loss),
            "validation_perplexity": math.exp(float(validation_loss)),
            "validation_batches": VALIDATION_EVAL_BATCHES,
            "validation_context_length": VALIDATION_CONTEXT_LENGTH,
            "validation_stride": VALIDATION_STRIDE,
            "validation_batch_size": VALIDATION_BATCH_SIZE,
        }]),
        step=step,
    )

    evaluation_efficiency = add_run_metadata(
        pd.DataFrame([{
            "model_id": model_id,
            "checkpoint": checkpoint_label,
            "fact_evaluation_seconds": fact_evaluation_seconds,
            "forget_set_evaluation_seconds": forget_set_evaluation_seconds,
            "validation_evaluation_seconds": validation_evaluation_seconds,
            "evaluation_compute_seconds": (
                fact_evaluation_seconds
                + forget_set_evaluation_seconds
                + validation_evaluation_seconds
            ),
        }]),
        step=step,
    )

    # ---------------------------------------------------------
    # 4. State-spezifische Rohdaten speichern
    # ---------------------------------------------------------
    results.to_csv(state_dir / "prompt_results.csv", index=False)
    results.to_json(
        state_dir / "prompt_results.jsonl",
        orient="records",
        lines=True,
        force_ascii=False,
    )
    primary.to_csv(state_dir / "primary_prompt_results.csv", index=False)
    fact_summary.to_csv(state_dir / "fact_summary.csv", index=False)
    group_summary.to_csv(state_dir / "group_summary.csv", index=False)
    forget_set_summary.to_csv(
        state_dir / "forget_set_loss_summary.csv",
        index=False,
    )
    forget_window_losses.to_csv(
        state_dir / "forget_set_window_losses.csv",
        index=False,
    )
    utility_summary.to_csv(state_dir / "utility_summary.csv", index=False)
    evaluation_efficiency.to_csv(
        state_dir / "evaluation_efficiency.csv",
        index=False,
    )

    if not alternative.empty:
        alternative.to_csv(
            state_dir / "alternative_target_results.csv",
            index=False,
        )
    if not counterfactual.empty:
        counterfactual.to_csv(
            state_dir / "counterfactual_results.csv",
            index=False,
        )

    write_json(
        state_dir / "metadata.json",
        {
            **COMMON_RUN_METADATA,
            "model_id": model_id,
            "unlearning_step": int(step),
            "checkpoint": checkpoint_label,
            "source_checkpoint": str(BASE_MODEL_CHECKPOINT),
            "validation_loss": float(validation_loss),
            "validation_batches": VALIDATION_EVAL_BATCHES,
            "prompt_target_evaluations": int(len(results)),
            "primary_prompt_evaluations": int(len(primary)),
            "forget_set_loss_summary": {
                "full_window_nll": float(
                    forget_set_summary.iloc[0]["full_window_nll"]
                ),
                "relation_target_nll": float(
                    forget_set_summary.iloc[0]["relation_target_nll"]
                ),
                "non_target_context_nll": float(
                    forget_set_summary.iloc[0]["non_target_context_nll"]
                ),
                "full_window_token_count": int(
                    forget_set_summary.iloc[0]["full_window_token_count"]
                ),
                "relation_target_token_count": int(
                    forget_set_summary.iloc[0]["relation_target_token_count"]
                ),
                "non_target_context_token_count": int(
                    forget_set_summary.iloc[0]["non_target_context_token_count"]
                ),
            },
            "evaluation_efficiency_seconds": {
                "fact": fact_evaluation_seconds,
                "forget_set": forget_set_evaluation_seconds,
                "validation": validation_evaluation_seconds,
                "total_compute": (
                    fact_evaluation_seconds
                    + forget_set_evaluation_seconds
                    + validation_evaluation_seconds
                ),
            },
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        },
    )

    print(
        f"Evaluation Step {step:04d}: "
        f"Validation Loss = {validation_loss:.6f} | "
        f"Relation NLL = "
        f"{forget_set_summary.iloc[0]['relation_target_nll']:.6f} | "
        f"Context NLL = "
        f"{forget_set_summary.iloc[0]['non_target_context_nll']:.6f}"
    )

    return {
        "results": results,
        "fact_summary": fact_summary,
        "group_summary": group_summary,
        "validation_loss": float(validation_loss),
        "utility_summary": utility_summary,
        "forget_set_summary": forget_set_summary,
        "forget_window_losses": forget_window_losses,
        "evaluation_efficiency": evaluation_efficiency,
    }


## 18. M0 als Step 0 mit derselben Pipeline evaluieren

In [ ]:
EVALUATION_STATE_BY_STEP = {}

EVALUATION_STATE_BY_STEP[0] = evaluate_and_save_model_state(
    model=MODEL,
    step=0,
    checkpoint_label=BASE_MODEL_CHECKPOINT.name,
)

print(
    "M0 Validation Loss: "
    f"{EVALUATION_STATE_BY_STEP[0]['validation_loss']:.6f}"
)


## 19. Gradient-Ascent-Funktionen

In [ ]:
def cycle_dataloader(data_loader):
    while True:
        for batch in data_loader:
            yield batch


def calc_forget_loss(
    *,
    input_batch,
    target_batch,
    model,
    device,
    loss_mode,
    loss_mask=None,
):
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)

    logits = model(input_batch)
    token_losses = F.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
        reduction="none",
    ).view_as(target_batch)

    if loss_mode == "full_window":
        return token_losses.mean(), int(token_losses.numel())

    if loss_mode == "masked":
        if loss_mask is None:
            raise ValueError(
                "Für loss_mode='masked' muss loss_mask gesetzt sein."
            )

        loss_mask = loss_mask.to(device=device, dtype=torch.bool)
        if loss_mask.shape != token_losses.shape:
            raise ValueError(
                f"Maskenform {tuple(loss_mask.shape)} != "
                f"Lossform {tuple(token_losses.shape)}"
            )

        active_tokens = int(loss_mask.sum().item())
        if active_tokens == 0:
            raise ValueError(
                "Die Forget-Maske enthält keine aktiven Positionen."
            )

        return token_losses[loss_mask].mean(), active_tokens

    raise ValueError(f"Unbekannter loss_mode: {loss_mode!r}")


def gradient_ascent_step(
    *,
    model,
    optimizer,
    input_batch,
    target_batch,
    device,
    gradient_clip_norm,
    loss_mode,
    loss_mask=None,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    forget_loss, active_loss_tokens = calc_forget_loss(
        input_batch=input_batch,
        target_batch=target_batch,
        model=model,
        device=device,
        loss_mode=loss_mode,
        loss_mask=loss_mask,
    )

    # Minimieren von -L_F entspricht Maximieren von L_F.
    ga_objective = -forget_loss
    ga_objective.backward()

    if gradient_clip_norm is not None:
        pre_clip_gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=gradient_clip_norm,
        )
        pre_clip_gradient_norm = float(pre_clip_gradient_norm.item())

        gradient_clipped = bool(
            pre_clip_gradient_norm > float(gradient_clip_norm)
        )
        clip_coefficient = min(
            1.0,
            float(gradient_clip_norm)
            / (pre_clip_gradient_norm + 1e-6),
        )
    else:
        pre_clip_gradient_norm = float("nan")
        gradient_clipped = False
        clip_coefficient = 1.0

    optimizer.step()

    return {
        "forget_loss": float(forget_loss.item()),
        "ga_objective": float(ga_objective.item()),
        "pre_clip_gradient_norm": pre_clip_gradient_norm,
        "gradient_clipped": gradient_clipped,
        "clip_coefficient": clip_coefficient,
        "active_loss_tokens": active_loss_tokens,
        "loss_mode": loss_mode,
    }


## 20. Gradient Ascent ausführen – Full-Window oder Masked

**Achtung:** Diese Zelle verändert `MODEL` in-place. Jeder neue Hyperparameterlauf muss wieder vom originalen M0-Checkpoint starten. Der AdamW-Optimizer wird für jeden Unlearning-Lauf neu initialisiert; ein Optimizer-State aus dem Vortraining wird nicht übernommen.

`gradient_ascent_step()` setzt das Modell während der Updates in den Trainingsmodus; Dropout bleibt daher während des Gradient Ascent aktiv.


In [ ]:
optimizer = torch.optim.AdamW(
    MODEL.parameters(),
    lr=UNLEARNING_CONFIG["learning_rate"],
    weight_decay=UNLEARNING_CONFIG["weight_decay"],
)

forget_iterator = cycle_dataloader(FORGET_LOADER)
training_history = []
cumulative_ga_update_seconds = 0.0

for step in range(1, UNLEARNING_CONFIG["max_steps"] + 1):
    input_batch, target_batch, window_ids = next(forget_iterator)

    loss_mask = None
    if LOSS_MODE == "masked":
        loss_mask = torch.stack([
            WINDOW_LOSS_MASKS[int(window_id)]
            for window_id in window_ids.tolist()
        ])

    # ---------------------------------------------------------
    # Effizienz: nur den eigentlichen GA-Update messen.
    # Evaluation, Checkpoint-I/O und CSV-I/O sind NICHT enthalten.
    # ---------------------------------------------------------
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        gpu_memory_before_mb = (
            torch.cuda.memory_allocated(DEVICE) / (1024 ** 2)
        )
    else:
        gpu_memory_before_mb = float("nan")

    ga_step_start = time.perf_counter()

    step_metrics = gradient_ascent_step(
        model=MODEL,
        optimizer=optimizer,
        input_batch=input_batch,
        target_batch=target_batch,
        device=DEVICE,
        gradient_clip_norm=UNLEARNING_CONFIG["gradient_clip_norm"],
        loss_mode=LOSS_MODE,
        loss_mask=loss_mask,
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

    ga_step_time_seconds = time.perf_counter() - ga_step_start
    cumulative_ga_update_seconds += ga_step_time_seconds

    if DEVICE.type == "cuda":
        peak_gpu_memory_allocated_mb = (
            torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
        )
        peak_gpu_memory_reserved_mb = (
            torch.cuda.max_memory_reserved(DEVICE) / (1024 ** 2)
        )
    else:
        peak_gpu_memory_allocated_mb = float("nan")
        peak_gpu_memory_reserved_mb = float("nan")

    training_history.append({
        **COMMON_RUN_METADATA,
        "unlearning_step": int(step),
        "window_ids": json.dumps(
            [int(x) for x in window_ids.tolist()]
        ),
        "learning_rate": float(optimizer.param_groups[0]["lr"]),
        "ga_step_time_seconds": ga_step_time_seconds,
        "cumulative_ga_update_seconds": cumulative_ga_update_seconds,
        "gpu_memory_allocated_before_mb": gpu_memory_before_mb,
        "peak_gpu_memory_allocated_mb": peak_gpu_memory_allocated_mb,
        "peak_gpu_memory_reserved_mb": peak_gpu_memory_reserved_mb,
        **step_metrics,
    })

    # Nach jedem Schritt aktualisieren, damit Logs bei einem Abbruch erhalten bleiben.
    pd.DataFrame(training_history).to_csv(
        UNLEARNING_RUN_DIR / "training_history.csv",
        index=False,
    )

    print(
        f"Step {step:02d} | "
        f"Mode: {LOSS_MODE:<11} | "
        f"Forget Loss: {step_metrics['forget_loss']:.4f} | "
        f"Loss Tokens: {step_metrics['active_loss_tokens']} | "
        f"Pre-clip Grad: {step_metrics['pre_clip_gradient_norm']:.4f} | "
        f"Clipped: {step_metrics['gradient_clipped']} | "
        f"GA Time: {ga_step_time_seconds:.3f}s"
    )

    checkpoint_label = f"in_memory_step_{step:04d}"

    if step in UNLEARNING_CONFIG["checkpoint_steps"]:
        checkpoint_path = CHECKPOINT_DIR / f"ga_step_{step:04d}.pth"
        save_unlearning_checkpoint(
            path=checkpoint_path,
            model=MODEL,
            step=step,
            source_checkpoint=BASE_MODEL_CHECKPOINT,
            unlearning_config=UNLEARNING_CONFIG,
            forget_set_id=FORGET_DOCUMENT["forget_set_id"],
        )
        checkpoint_label = checkpoint_path.name
        print("Checkpoint gespeichert:", checkpoint_path.name)

        if SAVE_RESUME_CHECKPOINT:
            save_resume_checkpoint(
                path=CHECKPOINT_DIR / "resume_latest.pth",
                model=MODEL,
                optimizer=optimizer,
                step=step,
                source_checkpoint=BASE_MODEL_CHECKPOINT,
                unlearning_config=UNLEARNING_CONFIG,
                forget_set_id=FORGET_DOCUMENT["forget_set_id"],
            )

    if step in UNLEARNING_CONFIG["evaluation_steps"]:
        EVALUATION_STATE_BY_STEP[step] = evaluate_and_save_model_state(
            model=MODEL,
            step=step,
            checkpoint_label=checkpoint_label,
        )

# Bei batch_size=1 und max_steps=16 muss jedes Forget-Window exakt einmal
# verarbeitet worden sein.
seen_windows = [
    window_id
    for row in training_history
    for window_id in json.loads(row["window_ids"])
]

assert len(seen_windows) == 16
assert len(set(seen_windows)) == 16
assert set(seen_windows) == set(FORGET_WINDOW_IDS)

print("✓ Jedes der 16 Forget-Windows wurde exakt einmal verwendet.")


In [ ]:
seen_windows = [
    wid
    for row in training_history
    for wid in json.loads(row["window_ids"])
]

assert len(seen_windows) == 16
assert len(set(seen_windows)) == 16
assert set(seen_windows) == set(FORGET_WINDOW_IDS)

print("✓ Jedes der 16 Forget-Windows wurde exakt einmal verwendet.")

## 21. M0 gegen alle evaluierten GA-Zustände vergleichen

In [ ]:
def concatenate_state_frames(key):
    frames = [
        EVALUATION_STATE_BY_STEP[step][key]
        for step in sorted(EVALUATION_STATE_BY_STEP)
    ]
    return pd.concat(frames, ignore_index=True)


# ============================================================
# 1. Canonical Long/Tidy Histories
# ============================================================
EVALUATION_HISTORY = concatenate_state_frames("results")
FACT_HISTORY = concatenate_state_frames("fact_summary")
GROUP_HISTORY = concatenate_state_frames("group_summary")
FORGET_SET_LOSS_HISTORY = concatenate_state_frames("forget_set_summary")
FORGET_SET_WINDOW_LOSS_HISTORY = concatenate_state_frames(
    "forget_window_losses"
)
EVALUATION_EFFICIENCY_HISTORY = concatenate_state_frames(
    "evaluation_efficiency"
)

EVALUATION_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "evaluation_history.csv",
    index=False,
)
FACT_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "fact_history.csv",
    index=False,
)
GROUP_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "group_history.csv",
    index=False,
)
FORGET_SET_WINDOW_LOSS_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "forget_set_window_loss_history.csv",
    index=False,
)
EVALUATION_EFFICIENCY_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "evaluation_efficiency_history.csv",
    index=False,
)

# ============================================================
# 2. Deltas der tokenbasierten Forget-Set-Losses gegen M0
# ============================================================
forget_baseline = FORGET_SET_LOSS_HISTORY.loc[
    FORGET_SET_LOSS_HISTORY["unlearning_step"] == 0
].iloc[0]

for metric in [
    "full_window_nll",
    "relation_target_nll",
    "non_target_context_nll",
]:
    FORGET_SET_LOSS_HISTORY[f"delta_{metric}"] = (
        FORGET_SET_LOSS_HISTORY[metric]
        - float(forget_baseline[metric])
    )

# Positiv bedeutet: relationstragende Tokens wurden stärker geschwächt
# als die übrigen Tokens derselben 16 Forget-Windows.
FORGET_SET_LOSS_HISTORY["relation_minus_context_delta"] = (
    FORGET_SET_LOSS_HISTORY["delta_relation_target_nll"]
    - FORGET_SET_LOSS_HISTORY["delta_non_target_context_nll"]
)

FORGET_SET_LOSS_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "forget_set_loss_history.csv",
    index=False,
)

# ============================================================
# 3. Prompt-genauer M0-vs.-GA-Vergleich im stabilen Long-Format
# ============================================================
comparison_frames = []
baseline_results = EVALUATION_STATE_BY_STEP[0]["results"]

for step in sorted(EVALUATION_STATE_BY_STEP):
    if step == 0:
        continue

    comparison = compare_evaluation_runs(
        baseline_results,
        EVALUATION_STATE_BY_STEP[step]["results"],
    )
    comparison = add_run_metadata(comparison, step=step)
    comparison_frames.append(comparison)

if comparison_frames:
    ALL_COMPARISONS = pd.concat(
        comparison_frames,
        ignore_index=True,
    )
    ALL_COMPARISONS.to_csv(
        UNLEARNING_RUN_DIR / "all_prompt_comparisons.csv",
        index=False,
    )

    display(
        ALL_COMPARISONS[
            ALL_COMPARISONS["group"] == "target"
        ][
            [
                "unlearning_step",
                "prompt_id",
                "target_type",
                "target",
                "delta_sequence_nll",
                "delta_first_token_rank",
                "geo_probability_ratio",
            ]
        ].sort_values(
            ["unlearning_step", "prompt_id", "target_type", "target"]
        )
    )
else:
    print("Keine Kandidaten-Evaluationen vorhanden.")

print("\nTokenbasierte Forget-Set-Historie:")
display(
    FORGET_SET_LOSS_HISTORY[
        [
            "unlearning_step",
            "full_window_nll",
            "relation_target_nll",
            "non_target_context_nll",
            "delta_full_window_nll",
            "delta_relation_target_nll",
            "delta_non_target_context_nll",
            "relation_minus_context_delta",
        ]
    ]
)


## 22. Utility-Verlauf

In [ ]:
utility_rows = []

for step in sorted(EVALUATION_STATE_BY_STEP):
    validation_loss = float(
        EVALUATION_STATE_BY_STEP[step]["validation_loss"]
    )

    utility_rows.append({
        **COMMON_RUN_METADATA,
        "unlearning_step": int(step),
        "validation_loss": validation_loss,
        "validation_perplexity": math.exp(validation_loss),
    })

UTILITY_HISTORY = pd.DataFrame(utility_rows)

m0_validation_loss = float(
    UTILITY_HISTORY.loc[
        UTILITY_HISTORY["unlearning_step"] == 0,
        "validation_loss",
    ].iloc[0]
)
m0_validation_perplexity = math.exp(m0_validation_loss)

UTILITY_HISTORY["delta_validation_loss"] = (
    UTILITY_HISTORY["validation_loss"]
    - m0_validation_loss
)
UTILITY_HISTORY["validation_perplexity_ratio"] = (
    UTILITY_HISTORY["validation_perplexity"]
    / m0_validation_perplexity
)

UTILITY_HISTORY.to_csv(
    UNLEARNING_RUN_DIR / "utility_history.csv",
    index=False,
)

display(
    UTILITY_HISTORY[
        [
            "unlearning_step",
            "validation_loss",
            "delta_validation_loss",
            "validation_perplexity",
            "validation_perplexity_ratio",
        ]
    ]
)


## 23. Finale Run-Metadaten

In [ ]:
TOTAL_RUN_WALL_TIME_SECONDS = time.perf_counter() - RUN_WALL_START_TIME

TRAINING_HISTORY = pd.DataFrame(training_history)

if TRAINING_HISTORY.empty:
    raise RuntimeError("training_history ist leer.")

total_ga_update_time_seconds = float(
    TRAINING_HISTORY["ga_step_time_seconds"].sum()
)
mean_ga_step_time_seconds = float(
    TRAINING_HISTORY["ga_step_time_seconds"].mean()
)
median_ga_step_time_seconds = float(
    TRAINING_HISTORY["ga_step_time_seconds"].median()
)

total_evaluation_compute_seconds = float(
    EVALUATION_EFFICIENCY_HISTORY["evaluation_compute_seconds"].sum()
)

if DEVICE.type == "cuda":
    peak_gpu_memory_allocated_mb = float(
        TRAINING_HISTORY["peak_gpu_memory_allocated_mb"].max()
    )
    peak_gpu_memory_reserved_mb = float(
        TRAINING_HISTORY["peak_gpu_memory_reserved_mb"].max()
    )
else:
    peak_gpu_memory_allocated_mb = float("nan")
    peak_gpu_memory_reserved_mb = float("nan")

EFFICIENCY_SUMMARY = pd.DataFrame([{
    **COMMON_RUN_METADATA,
    "ga_update_count": int(len(TRAINING_HISTORY)),
    "total_ga_update_time_seconds": total_ga_update_time_seconds,
    "mean_ga_step_time_seconds": mean_ga_step_time_seconds,
    "median_ga_step_time_seconds": median_ga_step_time_seconds,
    "ga_updates_per_second": (
        len(TRAINING_HISTORY) / total_ga_update_time_seconds
        if total_ga_update_time_seconds > 0
        else float("nan")
    ),
    "peak_gpu_memory_allocated_mb": peak_gpu_memory_allocated_mb,
    "peak_gpu_memory_reserved_mb": peak_gpu_memory_reserved_mb,
    "total_evaluation_compute_seconds": total_evaluation_compute_seconds,
    "total_run_wall_time_seconds": TOTAL_RUN_WALL_TIME_SECONDS,
}])

EFFICIENCY_SUMMARY.to_csv(
    UNLEARNING_RUN_DIR / "efficiency_summary.csv",
    index=False,
)

efficiency_dict = EFFICIENCY_SUMMARY.iloc[0].to_dict()

final_metadata = {
    "run_name": UNLEARNING_RUN_NAME,
    "source_model_id": SOURCE_MODEL_ID,
    "source_checkpoint": str(BASE_MODEL_CHECKPOINT),
    "experiment_id": EXPERIMENT_ID,
    "benchmark_version": facts_document.get("benchmark_version", "unknown"),
    "forget_set_id": FORGET_DOCUMENT["forget_set_id"],
    "mask_set_id": MASK_DOCUMENT["mask_set_id"],
    "forget_window_count": len(FORGET_WINDOW_IDS),
    "unlearning_method": UNLEARNING_METHOD,
    "unlearning_config": UNLEARNING_CONFIG,
    "parameter_count": parameter_count,
    "seed": SEED,
    "device": str(DEVICE),
    "m0_validation_loss": float(
        EVALUATION_STATE_BY_STEP[0]["validation_loss"]
    ),
    "evaluated_steps": sorted(
        int(x) for x in EVALUATION_STATE_BY_STEP
    ),
    "saved_checkpoint_steps": UNLEARNING_CONFIG["checkpoint_steps"],
    "efficiency": {
        key: value
        for key, value in efficiency_dict.items()
        if key not in COMMON_RUN_METADATA
    },
    "canonical_history_files": {
        "evaluation": "evaluation_history.csv",
        "facts": "fact_history.csv",
        "groups": "group_history.csv",
        "prompt_comparisons": "all_prompt_comparisons.csv",
        "utility": "utility_history.csv",
        "forget_set": "forget_set_loss_history.csv",
        "forget_set_windows": "forget_set_window_loss_history.csv",
        "training": "training_history.csv",
        "evaluation_efficiency": "evaluation_efficiency_history.csv",
        "efficiency_summary": "efficiency_summary.csv",
    },
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

write_json(
    UNLEARNING_RUN_DIR / "metadata.json",
    final_metadata,
)

print("\nEffizienz-Zusammenfassung:")
display(EFFICIENCY_SUMMARY)

print(
    "Unlearning-Run vollständig gespeichert unter:\n"
    f"{UNLEARNING_RUN_DIR}"
)


## 24. Interpretationshinweise

- Positives $\Delta\mathrm{NLL}=\mathrm{NLL}_{GA}-\mathrm{NLL}_{M0}$ bedeutet, dass die Zielsequenz unwahrscheinlicher geworden ist.
- Der Validation Loss wird während der Runs für alle Zustände auf denselben 50 Validation-Batches gemessen.
- `same_window` ist beim Full-Window-Modus besonders vorsichtig zu interpretieren, da alle Targetpositionen der Forget-Fenster direkt zum maximierten Loss beitragen.
- Die Pipeline implementiert Full-Window und Masked (token-selektives) Gradient Ascent. Gradient Difference ist nicht Bestandteil dieser Experimente.
